In [14]:
# Pilar3_Herencia_Polimorfismo.ipynb
# Autor: Eduardo Zambrano
# Este programa demuestra los conceptos de Herencia y Polimorfismo
# VERSIÓN INTERACTIVA - Permite crear nuevos tipos y empleados

from datetime import datetime
import os  # Para limpiar pantalla

# ============================================================================
# CLASE PADRE (SUPERCLASE)
# ============================================================================
class Empleado:
    """
    Clase Padre: Define la estructura base para todos los empleados.

    HERENCIA: Las clases hijas heredarán todos los atributos y métodos de esta clase.
    POLIMORFISMO: Los métodos pueden ser sobreescritos por las clases hijas
    para tener comportamientos diferentes.
    """

    def __init__(self, nombre, id_empleado, salario_base):
        """
        Constructor de la clase padre.
        """
        self.nombre = nombre
        self.id_empleado = id_empleado
        self.salario_base = salario_base
        self.fecha_ingreso = datetime.now()  # Se asigna automáticamente al crear

    def calcular_salario(self):
        """
        Método que será SOBREESCRITO (override) por las clases hijas.
        """
        return self.salario_base

    def mostrar_informacion(self):
        """
        Método que usa polimorfismo.
        """
        return f"ID: {self.id_empleado} | {self.nombre:<29} | Salario: ${self.calcular_salario():,.2f}"

    def tipo_empleado(self):
        """
        Método que será sobreescrito por las clases hijas.
        """
        return "Empleado General"


# ============================================================================
# CLASES HIJAS (SUBCLASES) PREDEFINIDAS
# ============================================================================

class EmpleadoTiempoCompleto(Empleado):
    """Empleado con contrato a tiempo completo."""

    def __init__(self, nombre, id_empleado, salario_base, bono_anual):
        super().__init__(nombre, id_empleado, salario_base)
        self.bono_anual = bono_anual

    def calcular_salario(self):
        return self.salario_base + (self.bono_anual / 12)

    def tipo_empleado(self):
        return "Tiempo Completo"


class EmpleadoPorHoras(Empleado):
    """Empleado que cobra por horas trabajadas."""

    def __init__(self, nombre, id_empleado, salario_por_hora, horas_trabajadas):
        super().__init__(nombre, id_empleado, 0)
        self.salario_por_hora = salario_por_hora
        self.horas_trabajadas = horas_trabajadas

    def calcular_salario(self):
        return self.salario_por_hora * self.horas_trabajadas

    def tipo_empleado(self):
        return "Por Horas"


class EmpleadoComision(Empleado):
    """Empleado que recibe comisión por ventas."""

    def __init__(self, nombre, id_empleado, salario_base, ventas_realizadas, porcentaje_comision):
        super().__init__(nombre, id_empleado, salario_base)
        self.ventas_realizadas = ventas_realizadas
        self.porcentaje_comision = porcentaje_comision

    def calcular_salario(self):
        comision = self.ventas_realizadas * (self.porcentaje_comision / 100)
        return self.salario_base + comision

    def tipo_empleado(self):
        return "Con Comisión"


# ============================================================================
# DICCIONARIO DE TIPOS DE EMPLEADO (PARA CRECIMIENTO DINÁMICO)
# ============================================================================
# Este diccionario almacena las clases de empleados disponibles
# ¡PUEDE CRECER! Cuando crees un nuevo tipo, se agregará aquí
tipos_empleado = {
    "1": {
        "nombre": "Tiempo Completo",
        "clase": EmpleadoTiempoCompleto,
        "descripcion": "Salario base + bono anual",
        "parametros": ["nombre", "id_empleado", "salario_base", "bono_anual"]
    },
    "2": {
        "nombre": "Por Horas",
        "clase": EmpleadoPorHoras,
        "descripcion": "Salario por hora * horas trabajadas",
        "parametros": ["nombre", "id_empleado", "salario_por_hora", "horas_trabajadas"]
    },
    "3": {
        "nombre": "Con Comisión",
        "clase": EmpleadoComision,
        "descripcion": "Salario base + % de ventas",
        "parametros": ["nombre", "id_empleado", "salario_base", "ventas_realizadas", "porcentaje_comision"]
    }
}

# Contador para el siguiente ID disponible
proximo_id = 1

def generar_id():
    """Genera un ID único para cada empleado"""
    global proximo_id
    id_generado = f"E{proximo_id:03d}"  # Formato E001, E002, etc.
    proximo_id += 1
    return id_generado


# ============================================================================
# FUNCIONES PARA CREAR NUEVOS TIPOS DE EMPLEADO
# ============================================================================

def crear_nuevo_tipo_empleado():
    """
    Permite al usuario crear una NUEVA CLASE de empleado en tiempo de ejecución.
    Esto es PROGRAMACIÓN DINÁMICA - Creamos clases sobre la marcha.
    """
    print("\n" + "="*60)
    print("🆕 CREAR NUEVO TIPO DE EMPLEADO")
    print("="*60)

    print("\n📝 Define el nuevo tipo de empleado:")
    nombre_tipo = input("Nombre del nuevo tipo (ej: 'Remoto', 'Practicante', 'etc'): ").strip()

    # Validar que no exista ya
    for tipo in tipos_empleado.values():
        if tipo["nombre"].lower() == nombre_tipo.lower():
            print(f"❌ El tipo '{nombre_tipo}' ya existe")
            input("Presione ENTER para continuar...")
            return None

    print("\n⚙️  Ahora define cómo se calcula su salario:")
    print("   Puedes usar los siguientes atributos:")
    print("   • salario_base - El salario base del empleado")
    print("   • Puedes crear nuevos atributos específicos")

    # Preguntar qué atributos específicos tendrá
    print("\n📋 Atributos específicos (aparte del salario_base):")
    atributos = []
    while True:
        atributo = input("   Nombre del atributo (ENTER para terminar): ").strip()
        if not atributo:
            break
        if atributo != "salario_base":  # Evitar duplicar salario_base
            atributos.append(atributo)

    # Preguntar la fórmula de cálculo
    print("\n🧮 Define la fórmula para calcular el salario:")
    print("   Puedes usar: salario_base, " + ", ".join(atributos) if atributos else "   (no hay atributos adicionales)")
    print("   Ejemplo: salario_base + (bono_anual / 12)")
    print("   Ejemplo: salario_base * 1.1 + comision")

    formula = input("\n✏️  Ingresa la fórmula: ").strip()

    # Crear la nueva clase DINÁMICAMENTE
    print("\n⚙️  Creando nueva clase...")

    # Creamos el método __init__ dinámicamente
    def init_method(self, nombre, id_empleado, salario_base, **kwargs):
        Empleado.__init__(self, nombre, id_empleado, salario_base)
        for attr in atributos:
            if attr in kwargs:
                setattr(self, attr, kwargs[attr])
            else:
                setattr(self, attr, 0)  # Valor por defecto

    # Creamos el método calcular_salario con la fórmula proporcionada
    def calcular_salario_method(self):
        try:
            # Usamos eval para ejecutar la fórmula (con precaución)
            # En un entorno real, deberías usar un parser más seguro
            return eval(formula, {"__builtins__": {}}, vars(self))
        except Exception as e:
            print(f"⚠️  Error en la fórmula: {e}")
            return self.salario_base

    # Creamos la nueva clase
    NuevaClase = type(
        f"Empleado{nombre_tipo.replace(' ', '')}",
        (Empleado,),
        {
            "__init__": init_method,
            "calcular_salario": calcular_salario_method,
            "tipo_empleado": lambda self: nombre_tipo,
            "atributos_especificos": atributos,
            "formula": formula
        }
    )

    # Agregar al diccionario de tipos
    nuevo_id = str(len(tipos_empleado) + 1)
    tipos_empleado[nuevo_id] = {
        "nombre": nombre_tipo,
        "clase": NuevaClase,
        "descripcion": f"Salario calculado como: {formula}",
        "parametros": ["nombre", "id_empleado", "salario_base"] + atributos
    }

    print(f"\n✅ ¡Nuevo tipo '{nombre_tipo}' creado exitosamente!")
    print(f"   ID asignado: {nuevo_id}")
    print(f"   Fórmula: {formula}")
    return nuevo_id


# ============================================================================
# FUNCIÓN PARA CREAR NUEVOS EMPLEADOS
# ============================================================================

def crear_nuevo_empleado():
    """
    Permite al usuario crear un nuevo empleado de cualquier tipo disponible.
    """
    print("\n" + "="*60)
    print("👤 CREAR NUEVO EMPLEADO")
    print("="*60)

    # Mostrar tipos disponibles
    print("\n📋 TIPOS DE EMPLEADO DISPONIBLES:")
    for id_tipo, tipo in tipos_empleado.items():
        print(f"   {id_tipo}. {tipo['nombre']} - {tipo['descripcion']}")

    # Seleccionar tipo
    try:
        opcion = input("\n🔷 Seleccione tipo de empleado: ").strip()

        if opcion not in tipos_empleado:
            print("❌ Tipo no válido")
            input("Presione ENTER para continuar...")
            return None

        tipo_seleccionado = tipos_empleado[opcion]

        print(f"\n--- CREANDO EMPLEADO TIPO: {tipo_seleccionado['nombre']} ---")

        # Solicitar datos básicos
        nombre = input("Nombre del empleado: ").strip()
        if not nombre:
            print("❌ El nombre no puede estar vacío")
            input("Presione ENTER...")
            return None

        # Generar ID automático
        id_empleado = generar_id()
        print(f"ID asignado: {id_empleado}")

        # Solicitar salario base (si aplica)
        salario_base = 0
        if "salario_base" in tipo_seleccionado["parametros"]:
            try:
                salario_base = float(input("Salario base: $"))
            except ValueError:
                print("❌ Valor inválido, se usará 0")
                salario_base = 0

        # Solicitar atributos específicos
        kwargs = {}
        for param in tipo_seleccionado["parametros"]:
            if param not in ["nombre", "id_empleado", "salario_base"]:
                try:
                    valor = float(input(f"{param}: $"))
                    kwargs[param] = valor
                except ValueError:
                    print(f"❌ Valor inválido para {param}, se usará 0")
                    kwargs[param] = 0

        # Crear el empleado según el tipo
        if tipo_seleccionado["nombre"] == "Tiempo Completo":
            empleado = EmpleadoTiempoCompleto(nombre, id_empleado, salario_base, kwargs.get("bono_anual", 0))
        elif tipo_seleccionado["nombre"] == "Por Horas":
            empleado = EmpleadoPorHoras(nombre, id_empleado, kwargs.get("salario_por_hora", 0), kwargs.get("horas_trabajadas", 0))
        elif tipo_seleccionado["nombre"] == "Con Comisión":
            empleado = EmpleadoComision(nombre, id_empleado, salario_base, kwargs.get("ventas_realizadas", 0), kwargs.get("porcentaje_comision", 0))
        else:
            # Para tipos creados dinámicamente
            clase = tipo_seleccionado["clase"]
            empleado = clase(nombre, id_empleado, salario_base, **kwargs)

        print(f"\n✅ ¡Empleado creado exitosamente!")
        print(f"   {empleado.mostrar_informacion()}")
        print(f"   Tipo: {empleado.tipo_empleado()}")

        input("\nPresione ENTER para continuar...")
        return empleado

    except Exception as e:
        print(f"❌ Error al crear empleado: {e}")
        input("Presione ENTER para continuar...")
        return None


# ============================================================================
# FUNCIÓN POLIMÓRFICA
# ============================================================================

def procesar_nomina(empleados):
    """
    Esta función DEMUESTRA POLIMORFISMO.
    """
    print("\n" + "="*70)
    print("💰 PROCESANDO NÓMINA - DEMOSTRACIÓN DE POLIMORFISMO")
    print("="*70)

    if not empleados:
        print("📭 No hay empleados para procesar")
        return 0

    total_nomina = 0

    for i, empleado in enumerate(empleados, 1):
        salario = empleado.calcular_salario()
        print(f"{i:2d}. {empleado.mostrar_informacion()} | Tipo: {empleado.tipo_empleado()}")
        total_nomina += salario

    print("-"*70)
    print(f"💰 TOTAL NÓMINA: ${total_nomina:,.2f} | Empleados: {len(empleados)}")
    print("="*70)
    return total_nomina


def mostrar_todos_empleados(empleados):
    """Muestra todos los empleados en formato tabla"""
    if not empleados:
        print("\n📭 No hay empleados registrados")
        return

    print("\n" + "="*80)
    print("📋 LISTA COMPLETA DE EMPLEADOS")
    print("="*80)
    print(f"{'#':<3} | {'ID':<6} | {'NOMBRE':<29} | {'TIPO':<15} | {'SALARIO':>12}")
    print("-"*80)

    for i, emp in enumerate(empleados, 1):
        print(f"{i:<3} | {emp.id_empleado:<6} | {emp.nombre:<30} | {emp.tipo_empleado():<15} | ${emp.calcular_salario():>11,.2f}")

    print("="*80)


def mostrar_tipos_disponibles():
    """Muestra todos los tipos de empleado disponibles"""
    print("\n" + "="*60)
    print("📚 TIPOS DE EMPLEADO DISPONIBLES")
    print("="*60)

    for id_tipo, tipo in tipos_empleado.items():
        print(f"\n{id_tipo}. {tipo['nombre']}")
        print(f"   📝 {tipo['descripcion']}")
        print(f"   📋 Parámetros: {', '.join(tipo['parametros'])}")


def limpiar_pantalla():
    """Limpia la pantalla"""
    os.system('cls' if os.name == 'nt' else 'clear')


# ============================================================================
# MENÚ PRINCIPAL
# ============================================================================

def menu_principal():
    """Función principal con menú interactivo"""

    # Lista donde se guardarán TODOS los empleados
    empleados = []

    # Creamos algunos empleados de ejemplo
    print("📌 Cargando empleados de ejemplo...")
    empleados.append(EmpleadoTiempoCompleto("Ana López", "E001", 3000, 6000))
    empleados.append(EmpleadoPorHoras("Carlos Ruiz", "E002", 25, 160))
    empleados.append(EmpleadoComision("María García", "E003", 2000, 50000, 5))
    print(f"✅ {len(empleados)} empleados cargados\n")
    input("Presione ENTER para continuar...")

    while True:
        limpiar_pantalla()

        print("\n" + "="*70)
        print("🏢  SISTEMA DE NÓMINA - HERENCIA Y POLIMORFISMO")
        print("="*70)
        print(f"📊 Empleados registrados: {len(empleados)}")
        print("-"*70)
        print("1. 👤 CREAR NUEVO EMPLEADO")
        print("2. 🆕 CREAR NUEVO TIPO DE EMPLEADO")
        print("3. 📋 MOSTRAR TODOS LOS EMPLEADOS")
        print("4. 📚 MOSTRAR TIPOS DISPONIBLES")
        print("5. 💰 PROCESAR NÓMINA")
        print("6. ❌ SALIR")
        print("-"*70)

        opcion = input("🔷 Seleccione una opción: ").strip()

        if opcion == '1':
            # CREAR NUEVO EMPLEADO
            nuevo_empleado = crear_nuevo_empleado()
            if nuevo_empleado:
                empleados.append(nuevo_empleado)
                print(f"\n✅ Empleado agregado. Total: {len(empleados)}")
                input("Presione ENTER...")

        elif opcion == '2':
            # CREAR NUEVO TIPO DE EMPLEADO
            crear_nuevo_tipo_empleado()
            input("Presione ENTER...")

        elif opcion == '3':
            # MOSTRAR TODOS LOS EMPLEADOS
            mostrar_todos_empleados(empleados)
            input("\nPresione ENTER...")

        elif opcion == '4':
            # MOSTRAR TIPOS DISPONIBLES
            mostrar_tipos_disponibles()
            input("\nPresione ENTER...")

        elif opcion == '5':
            # PROCESAR NÓMINA
            procesar_nomina(empleados)
            input("\nPresione ENTER...")

        elif opcion == '6':
            print("\n👋 ¡Gracias por usar el sistema de nómina!")
            break

        else:
            print("❌ Opción no válida")
            input("Presione ENTER...")


# ============================================================================
# EJECUTAR EL PROGRAMA
# ============================================================================

if __name__ == "__main__":
    menu_principal()

📌 Cargando empleados de ejemplo...
✅ 3 empleados cargados

Presione ENTER para continuar...

🏢  SISTEMA DE NÓMINA - HERENCIA Y POLIMORFISMO
📊 Empleados registrados: 3
----------------------------------------------------------------------
1. 👤 CREAR NUEVO EMPLEADO
2. 🆕 CREAR NUEVO TIPO DE EMPLEADO
3. 📋 MOSTRAR TODOS LOS EMPLEADOS
4. 📚 MOSTRAR TIPOS DISPONIBLES
5. 💰 PROCESAR NÓMINA
6. ❌ SALIR
----------------------------------------------------------------------
🔷 Seleccione una opción: 5

💰 PROCESANDO NÓMINA - DEMOSTRACIÓN DE POLIMORFISMO
 1. ID: E001 | Ana López                     | Salario: $3,500.00 | Tipo: Tiempo Completo
 2. ID: E002 | Carlos Ruiz                   | Salario: $4,000.00 | Tipo: Por Horas
 3. ID: E003 | María García                  | Salario: $4,500.00 | Tipo: Con Comisión
----------------------------------------------------------------------
💰 TOTAL NÓMINA: $12,000.00 | Empleados: 3

Presione ENTER...

🏢  SISTEMA DE NÓMINA - HERENCIA Y POLIMORFISMO
📊 Empleados reg